# ЛР-03: Бюджет топливного резерва

## Worked example: military 02

Это полностью разобранный example по duality и анализу чувствительности. Он показывает полный цикл от прямой модели до сценарных пересчётов.

## 1. Исходный кейс

Здесь duality помогает понять, чего не хватает для устойчивого резерва топлива.

### Ограничения ресурсов

| Ресурс | Лимит |
| --- | --- |
| Бюджет | 95 |
| Персонал | 58 |
| Ёмкости хранения | 47 |

### Программы

| Программа | Эффект | Бюджет | Трудозатраты | Операционная ёмкость |
| --- | --- | --- | --- | --- |
| Северный резерв | 87 | 36 | 17 | 16 |
| Южный резерв | 83 | 32 | 15 | 14 |
| Мобильные топливозаправщики | 76 | 24 | 18 | 11 |
| Автоматизация учёта топлива | 69 | 18 | 8 | 9 |

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

def solve_primal(effects, A_ub, b_ub):
    result = linprog(-effects, A_ub=A_ub, b_ub=b_ub, bounds=[(0, 1)] * len(effects), method='highs')
    if not result.success:
        raise RuntimeError(result.message)
    shadow_prices = -result.ineqlin.marginals
    return result, shadow_prices

def solve_dual(effects, A_ub, b_ub):
    m, n = A_ub.shape
    c_dual = np.concatenate([b_ub, np.ones(n)])
    A_dual = -np.hstack([A_ub.T, np.eye(n)])
    b_dual = -effects
    result = linprog(c_dual, A_ub=A_dual, b_ub=b_dual, bounds=[(0, None)] * (m + n), method='highs')
    if not result.success:
        raise RuntimeError(result.message)
    return result

def rerun_with_resource_change(effects, A_ub, b_ub, resource_index, delta):
    new_b = b_ub.copy()
    new_b[resource_index] += delta
    result, _ = solve_primal(effects, A_ub, new_b)
    return new_b, result


In [2]:
program_names = ['Северный резерв', 'Южный резерв', 'Мобильные топливозаправщики', 'Автоматизация учёта топлива']
resource_names = ['Бюджет', 'Персонал', 'Ёмкости хранения']
effects = np.array([87, 83, 76, 69], dtype=float)
A_ub = np.array([[36, 32, 24, 18], [17, 15, 18, 8], [16, 14, 11, 9]], dtype=float)
b_ub = np.array([95, 58, 47], dtype=float)

primal_result, shadow_prices = solve_primal(effects, A_ub, b_ub)
dual_result = solve_dual(effects, A_ub, b_ub)

plan_df = pd.DataFrame({
    'программа': program_names,
    'x*': np.round(primal_result.x, 4),
    'эффект на единицу': effects,
})
resources_df = pd.DataFrame({
    'ресурс': resource_names,
    'лимит': b_ub,
    'slack': np.round(primal_result.slack, 4),
    'shadow_price': np.round(shadow_prices, 4),
    'binding': np.isclose(primal_result.slack, 0.0),
})

print('Оптимальный эффект (primal):', round(-primal_result.fun, 4))
print('Оптимальное значение dual:', round(dual_result.fun, 4))
print()
print('Оптимальный план:')
display(plan_df)
print('Ресурсный разбор:')
display(resources_df)


Оптимальный эффект (primal): 278.75
Оптимальное значение dual: 278.75

Оптимальный план:


,программа,x*,эффект на единицу
0,Северный резерв,0.5833,87.0
1,Южный резерв,1.0000,83.0
2,Мобильные топливозаправщики,1.0000,76.0
3,Автоматизация учёта топлива,1.0000,69.0


Ресурсный разбор:


,ресурс,лимит,slack,shadow_price,binding
0,Бюджет,95.0,0.0000,2.4167,True
1,Персонал,58.0,7.0833,0.0000,False
2,Ёмкости хранения,47.0,3.6667,0.0000,False


In [3]:
scenario_rows = []
for label, resource_index, delta in [
    ('Бюджет +3', 0, 3),
    ('Ёмкости хранения +2', 2, 2),
]:
    new_b, new_result = rerun_with_resource_change(effects, A_ub, b_ub, resource_index, delta)
    predicted = shadow_prices[resource_index] * delta
    actual = (-new_result.fun) - (-primal_result.fun)
    scenario_rows.append({
        'сценарий': label,
        'ресурс': resource_names[resource_index],
        'delta': delta,
        'прогноз по shadow price': round(predicted, 4),
        'факт после пересчёта': round(actual, 4),
        'разница': round(actual - predicted, 4),
    })

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)


,сценарий,ресурс,delta,прогноз по shadow price,факт после пересчёта,разница
0,Бюджет +3,Бюджет,3,7.25,7.25,0.0
1,Ёмкости хранения +2,Ёмкости хранения,2,0.00,0.00,0.0


In [4]:
objective_label = 'Топливозаправщики +7 к эффекту'
objective_index = 2
objective_delta = 7

new_effects = effects.copy()
new_effects[objective_index] += objective_delta
objective_result, _ = solve_primal(new_effects, A_ub, b_ub)
objective_df = pd.DataFrame({
    'сценарий': [objective_label],
    'новый оптимальный эффект': [round(-objective_result.fun, 4)],
    'изменение эффекта': [round((-objective_result.fun) - (-primal_result.fun), 4)],
    'новое значение программы': [round(objective_result.x[objective_index], 4)],
})
display(objective_df)


,сценарий,новый оптимальный эффект,изменение эффекта,новое значение программы
0,Топливозаправщики +7 к эффекту,285.75,7.0,1.0


## 2. Что важно проговорить в выводе

- какие ограничения оказались binding и почему именно они держат оптимум;
- какой ресурс имеет наибольшую теневую цену и что это означает содержательно;
- насколько хорошо совпал прогноз по shadow price с фактическим пересчётом;
- меняется ли структура плана при небольших изменениях `b` и `c`.